# Tutorial: DocIR And Shared Editing API Demo

Audience:
- Engineers exploring `document_processor` from the `doc_processor` workspace.

Prerequisites:
- The `doc_processor` environment is installed and can import `document_processor`.
- You have a sample `.docx`, `.hwpx`, or `.hwp` file available.

Learning goals:
- Parse the same document from a path and from bytes.
- Inspect `DocIR` paragraphs, runs, and stable unit ids.
- Use the stateless context, target-discovery, edit, and review APIs.
- Compare in-memory, path-backed, and bytes-backed edit flows.
- Use exact-text annotations with optional `occurrence_index`, then inspect backend-resolved spans.
- Visualize parser LangGraph labels as color-coded review annotations.


## Outline

1. Resolve local sample paths and helpers.
2. Parse a real document into `DocIR`.
3. Inspect exact context and editable targets.
4. Render HTML preview and annotated review output.
5. Apply edits in three modes: `DocIR`, native file, and bytes.
6. Run the parser LangGraph on another sample and export category-colored review HTML.


In [1]:
from __future__ import annotations

from collections import Counter
from pathlib import Path

from document_processor import (
    ApplyTextEditsRequest,
    DocIR,
    DocumentInput,
    GetDocumentContextRequest,
    ListEditableTargetsRequest,
    RenderReviewHtmlRequest,
    TextAnnotation,
    TextEdit,
    apply_text_edits,
    get_document_context,
    list_editable_targets,
    render_review_html,
)
from doc_processor import ParagraphCategory, ParserConfig, RelevanceMode, run_parser

PROJECT_DIR = Path.cwd()
while PROJECT_DIR != PROJECT_DIR.parent and not (PROJECT_DIR / "pyproject.toml").exists():
    PROJECT_DIR = PROJECT_DIR.parent

TESTS_DIR = PROJECT_DIR / "tests"

SAMPLE_DOC = TESTS_DIR / "doc_samples" / "new_test" / "01. 대중문화예술분야 연습생 표준계약서.hwpx"
# SAMPLE_DOC = TESTS_DIR / "doc_samples" / "new_test" / "02. 청소년 대중문화예술인 표준 부속합의서.hwpx"
# SAMPLE_DOC = TESTS_DIR / "doc_samples" / "new_test" / "style_test_sample.docx"

OUTPUT_DIR = TESTS_DIR / "results" / "docir_api_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert SAMPLE_DOC.exists(), SAMPLE_DOC
SAMPLE_DOC_TYPE = SAMPLE_DOC.suffix.lower().lstrip(".")
SAMPLE_WRITEBACK_SUFFIX = ".hwpx" if SAMPLE_DOC_TYPE == "hwp" else SAMPLE_DOC.suffix.lower()


def pick_demo_paragraph(doc: DocIR):
    for paragraph in doc.paragraphs:
        if paragraph.runs and not paragraph.tables and not paragraph.images and paragraph.text.strip():
            for run in paragraph.runs:
                if run.text.strip():
                    return paragraph, run
    raise ValueError("Could not find a simple text paragraph/run pair for the demo.")


def paragraph_by_id(doc: DocIR, unit_id: str):
    for paragraph in doc.paragraphs:
        if paragraph.unit_id == unit_id:
            return paragraph
    raise KeyError(unit_id)


def run_by_id(doc: DocIR, unit_id: str):
    for paragraph in doc.paragraphs:
        for run in paragraph.runs:
            if run.unit_id == unit_id:
                return run
    raise KeyError(unit_id)


PARSER_CATEGORY_COLORS = {
    ParagraphCategory.TITLE: "#D9EAD3",
    ParagraphCategory.PREAMBLE: "#FFF2CC",
    ParagraphCategory.CLAUSE_HEADING: "#F4CCCC",
    ParagraphCategory.CLAUSE_BODY: "#FCE5CD",
    ParagraphCategory.SUBCLAUSE_HEADING: "#D9D2E9",
    ParagraphCategory.SUBCLAUSE_BODY: "#CFE2F3",
    ParagraphCategory.INPUT_BLOCK: "#D0E0E3",
    ParagraphCategory.APPENDIX: "#EAD1DC",
    ParagraphCategory.HEADER: "#EFEFEF",
    ParagraphCategory.FOOTER: "#EFEFEF",
    ParagraphCategory.OTHER: "#E6E6E6",
    ParagraphCategory.BOUNDARY_SUSPECT: "#F9CB9C",
}


def parser_category_label(category: ParagraphCategory | None) -> str:
    if category is None:
        return "Unlabeled"
    return category.value.replace("_", " ").title()


def parser_category_color(category: ParagraphCategory | None) -> str:
    if category is None:
        return "#E6E6E6"
    return PARSER_CATEGORY_COLORS.get(category, "#E6E6E6")


def occurrence_index_for_span(text: str, selected_text: str, start: int) -> int | None:
    matches = []
    search_from = 0
    while True:
        index = text.find(selected_text, search_from)
        if index < 0:
            break
        matches.append(index)
        search_from = index + 1
    if not matches:
        raise ValueError(f"Could not find {selected_text!r} in paragraph text.")
    occurrence_index = matches.index(start)
    return occurrence_index if len(matches) > 1 else None


def parser_annotations_from_doc(doc: DocIR):
    annotations = []
    skipped = []
    for paragraph in doc.paragraphs:
        parser_meta = paragraph.meta.parser if paragraph.meta and paragraph.meta.parser else None
        if parser_meta is None or not paragraph.text.strip():
            continue
        if paragraph.tables or paragraph.images:
            skipped.append(paragraph.unit_id)
            continue

        if parser_meta.spans:
            for span in parser_meta.spans:
                selected_text = paragraph.text[span.start:span.end]
                if not selected_text:
                    continue
                occurrence_index = occurrence_index_for_span(paragraph.text, selected_text, span.start)
                note_parts = [f"category={span.kind.value}"]
                if span.clause_no:
                    note_parts.append(f"clause={span.clause_no}")
                if span.subclause_no:
                    note_parts.append(f"subclause={span.subclause_no}")
                annotations.append(
                    TextAnnotation(
                        target_kind="paragraph",
                        target_unit_id=paragraph.unit_id,
                        selected_text=selected_text,
                        occurrence_index=occurrence_index,
                        label=parser_category_label(span.kind),
                        color=parser_category_color(span.kind),
                        note=" | ".join(note_parts),
                    )
                )
            continue

        if parser_meta.category is None:
            continue

        note_parts = [f"category={parser_meta.category.value}"]
        if parser_meta.clause_no:
            note_parts.append(f"clause={parser_meta.clause_no}")
        if parser_meta.subclause_no:
            note_parts.append(f"subclause={parser_meta.subclause_no}")
        if parser_meta.boundary_suspect:
            note_parts.append("boundary_suspect=true")
        annotations.append(
            TextAnnotation(
                target_kind="paragraph",
                target_unit_id=paragraph.unit_id,
                label=parser_category_label(parser_meta.category),
                color=parser_category_color(parser_meta.category),
                note=" | ".join(note_parts),
            )
        )

    return annotations, skipped

SAMPLE_DOC


PosixPath('/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/new_test/01. 대중문화예술분야 연습생 표준계약서.hwpx')

## Step 1 - Parse the same document from a path and from bytes

This shows that `DocIR.from_file(...)` supports both native paths and upload-style bytes.


In [2]:
sample_bytes = SAMPLE_DOC.read_bytes()

doc_from_path = DocIR.from_file(SAMPLE_DOC)
doc_from_bytes = DocIR.from_file(sample_bytes, doc_type=SAMPLE_DOC_TYPE)

demo_paragraph, demo_run = pick_demo_paragraph(doc_from_path)

summary = {
    "sample_doc": str(SAMPLE_DOC.relative_to(TESTS_DIR)),
    "source_doc_type": doc_from_path.source_doc_type,
    "paragraph_count": len(doc_from_path.paragraphs),
    "page_count": len(doc_from_path.pages),
    "path_vs_bytes_same_first_paragraph": doc_from_path.paragraphs[0].text == doc_from_bytes.paragraphs[0].text,
    "demo_paragraph_unit_id": demo_paragraph.unit_id,
    "demo_run_unit_id": demo_run.unit_id,
}

first_paragraphs = [
    {
        "unit_id": paragraph.unit_id,
        "text": paragraph.text,
        "run_ids": [run.unit_id for run in paragraph.runs],
    }
    for paragraph in doc_from_path.paragraphs[:3]
]

summary, first_paragraphs


({'sample_doc': 'doc_samples/new_test/01. 대중문화예술분야 연습생 표준계약서.hwpx',
  'source_doc_type': 'hwpx',
  'paragraph_count': 120,
  'page_count': 6,
  'path_vs_bytes_same_first_paragraph': True,
  'demo_paragraph_unit_id': 's1.p1',
  'demo_run_unit_id': 's1.p1.r2'},
 [{'unit_id': 's1.p1', 'text': '[별표1]', 'run_ids': ['s1.p1.r1', 's1.p1.r2']},
  {'unit_id': 's1.p2', 'text': '대중문화예술분야 연습생 표준계약서', 'run_ids': ['s1.p2.r1']},
  {'unit_id': 's1.p3',
   'text': '문화체육관광부고시 \n제2025-0069호\n(2026. 01. 01. 개정)',
   'run_ids': []}])

## Step 2 - Inspect exact context and discover safe edit targets

The stateless API accepts `DocumentInput`, so the caller can provide a path, bytes, or a preloaded `DocIR`.


In [3]:
context = get_document_context(
    GetDocumentContextRequest(
        document=DocumentInput(source_path=str(SAMPLE_DOC)),
        unit_ids=[demo_run.unit_id],
        before=0,
        after=1,
        include_runs=True,
    )
)

targets = list_editable_targets(
    ListEditableTargetsRequest(
        document=DocumentInput(doc_ir=doc_from_path),
        unit_ids=[demo_paragraph.unit_id],
        target_kinds=["paragraph", "run"],
        include_child_runs=True,
    )
)

context_summary = [
    {
        "unit_id": paragraph.unit_id,
        "text": paragraph.text,
        "runs": [(run.unit_id, run.text) for run in paragraph.runs],
    }
    for paragraph in context.paragraphs
]

target_summary = [
    {
        "target_kind": target.target_kind,
        "target_unit_id": target.target_unit_id,
        "current_text": target.current_text,
        "writable": target.writable,
    }
    for target in targets.targets
]

context_summary, target_summary


([{'unit_id': 's1.p1',
   'text': '[별표1]',
   'runs': [('s1.p1.r1', ''), ('s1.p1.r2', '[별표1]')]},
  {'unit_id': 's1.p2',
   'text': '대중문화예술분야 연습생 표준계약서',
   'runs': [('s1.p2.r1', '대중문화예술분야 연습생 표준계약서')]}],
 [{'target_kind': 'paragraph',
   'target_unit_id': 's1.p1',
   'current_text': '[별표1]',
   'writable': True},
  {'target_kind': 'run',
   'target_unit_id': 's1.p1.r1',
   'current_text': '',
   'writable': True},
  {'target_kind': 'run',
   'target_unit_id': 's1.p1.r2',
   'current_text': '[별표1]',
   'writable': True}])

## Step 3 - Render HTML preview and annotated review output

`DocIR.to_html(...)` gives a clean preview, and `render_review_html(...)` adds review annotations anchored by unit id and exact selected text.

If the same substring repeats inside a target, use `occurrence_index` instead of asking the model to count character offsets.


In [4]:
paragraph_selection = demo_paragraph.text[: min(20, len(demo_paragraph.text))]
run_selection = demo_run.text[: min(8, len(demo_run.text))]

preview_html = doc_from_path.to_html(title="DocIR Preview")
preview_path = OUTPUT_DIR / "preview.html"
preview_path.write_text(preview_html, encoding="utf-8")

review = render_review_html(
    RenderReviewHtmlRequest(
        document=DocumentInput(doc_ir=doc_from_path),
        annotations=[
            TextAnnotation(
                target_kind="paragraph",
                target_unit_id=demo_paragraph.unit_id,
                selected_text=paragraph_selection,
                label="Paragraph span",
                color="#FFD966",
                note="Paragraph-level review anchor",
            ),
            TextAnnotation(
                target_kind="run",
                target_unit_id=demo_run.unit_id,
                selected_text=run_selection,
                label="Run span",
                color="#99EEFF",
                note="Run-level review anchor",
            ),
        ],
        title="DocIR Review Demo",
    )
)

ambiguous_doc = DocIR.from_mapping({"s1.p1.r1": "Beta Beta Beta"}, source_doc_type="docx")
ambiguous_review = render_review_html(
    RenderReviewHtmlRequest(
        document=DocumentInput(doc_ir=ambiguous_doc),
        annotations=[
            TextAnnotation(
                target_kind="run",
                target_unit_id="s1.p1.r1",
                selected_text="Beta",
                occurrence_index=1,
                label="Second Beta",
                color="#C9DAF8",
                note="Repeated text uses occurrence_index, not raw offsets.",
            )
        ],
        title="Repeated Text Demo",
    )
)

review_path = OUTPUT_DIR / "review.html"
review_path.write_text(review.html or "", encoding="utf-8")

{
    "preview_path": str(preview_path.relative_to(TESTS_DIR)),
    "review_path": str(review_path.relative_to(TESTS_DIR)),
    "annotation_request": [
        {
            "target_kind": "paragraph",
            "target_unit_id": demo_paragraph.unit_id,
            "selected_text": paragraph_selection,
        },
        {
            "target_kind": "run",
            "target_unit_id": demo_run.unit_id,
            "selected_text": run_selection,
        },
    ],
    "resolved_annotations": [item.model_dump() for item in review.resolved_annotations],
    "repeated_text_example": ambiguous_review.resolved_annotations[0].model_dump(),
}


{'preview_path': 'results/docir_api_demo/preview.html',
 'review_path': 'results/docir_api_demo/review.html',
 'annotation_request': [{'target_kind': 'paragraph',
   'target_unit_id': 's1.p1',
   'selected_text': '[별표1]'},
  {'target_kind': 'run',
   'target_unit_id': 's1.p1.r2',
   'selected_text': '[별표1]'}],
 'resolved_annotations': [{'target_kind': 'paragraph',
   'target_unit_id': 's1.p1',
   'selected_text': '[별표1]',
   'occurrence_index': 0,
   'start': 0,
   'end': 5,
   'label': 'Paragraph span',
   'color': '#FFD966',
   'note': 'Paragraph-level review anchor'},
  {'target_kind': 'run',
   'target_unit_id': 's1.p1.r2',
   'selected_text': '[별표1]',
   'occurrence_index': 0,
   'start': 0,
   'end': 5,
   'label': 'Run span',
   'color': '#99EEFF',
   'note': 'Run-level review anchor'}],
 'repeated_text_example': {'target_kind': 'run',
  'target_unit_id': 's1.p1.r1',
  'selected_text': 'Beta',
  'occurrence_index': 1,
  'start': 5,
  'end': 9,
  'label': 'Second Beta',
  'color'

## Step 4 - Apply edits in three modes

This cell demonstrates:

- `DocIR`-only in-memory editing
- path-backed native file editing
- bytes-backed editing that returns edited bytes


In [8]:
in_memory_result = apply_text_edits(
    ApplyTextEditsRequest(
        document=DocumentInput(doc_ir=doc_from_path),
        edits=[
            TextEdit(
                target_kind="paragraph",
                target_unit_id=demo_paragraph.unit_id,
                expected_text=demo_paragraph.text,
                new_text=f"{demo_paragraph.text} [DocIR demo edit]",
                reason="Show in-memory paragraph editing",
            )
        ],
    )
)

native_output = OUTPUT_DIR / f"{SAMPLE_DOC.stem}_native_edit{SAMPLE_WRITEBACK_SUFFIX}"
native_result = apply_text_edits(
    ApplyTextEditsRequest(
        document=DocumentInput(source_path=str(SAMPLE_DOC)),
        edits=[
            TextEdit(
                target_kind="run",
                target_unit_id=demo_run.unit_id,
                expected_text=demo_run.text,
                new_text=f"[native edited here -> {demo_run.text.rstrip()} <- test]",
                reason="Show native file write-back",
            )
        ],
        output_path=str(native_output),
        return_doc_ir=True,
    )
)

bytes_result = apply_text_edits(
    ApplyTextEditsRequest(
        document=DocumentInput(source_bytes=sample_bytes, source_name=SAMPLE_DOC.name),
        edits=[
            TextEdit(
                target_kind="run",
                target_unit_id=demo_run.unit_id,
                expected_text=demo_run.text,
                new_text=f"{demo_run.text.rstrip()} [bytes]",
                reason="Show bytes-backed write-back",
            )
        ],
        return_doc_ir=True,
    )
)

{
    "in_memory_updated_paragraph": paragraph_by_id(in_memory_result.updated_doc_ir, demo_paragraph.unit_id).text,
    "native_output_path": str(Path(native_result.output_path).relative_to(TESTS_DIR)),
    "native_updated_run": run_by_id(native_result.updated_doc_ir, demo_run.unit_id).text,
    "bytes_output_filename": bytes_result.output_filename,
    "bytes_output_size": len(bytes_result.output_bytes or b""),
    "bytes_updated_run": run_by_id(bytes_result.updated_doc_ir, demo_run.unit_id).text,
}


{'in_memory_updated_paragraph': '[별표1] [DocIR demo edit]',
 'native_output_path': 'results/docir_api_demo/01. 대중문화예술분야 연습생 표준계약서_native_edit.hwpx',
 'native_updated_run': '[native edited here -> [별표1] <- test]',
 'bytes_output_filename': '01. 대중문화예술분야 연습생 표준계약서_edited.hwpx',
 'bytes_output_size': 63757,
 'bytes_updated_run': '[별표1] [bytes]'}

## Pitfalls, extensions, and one quick exercise

Common pitfalls:

- `expected_text` must match the current text exactly.
- `selected_text` must also match exactly; if it repeats inside the target, set `occurrence_index`.
- `DocumentInput(doc_ir=...)` does not produce a native file by itself.
- Paragraph edits are rejected when the paragraph contains tables or images.

Extensions:

- Switch `SAMPLE_DOC` to an `.hwpx` or `.hwp` sample and rerun the notebook.
- Replace the run edit with a paragraph edit and compare the modified run distribution.
- The final section visualizes parser labels as review annotations; turn on parser LLM review there if you want the reviewed path.

Exercise:

- Change `ALT_SAMPLE_DOC` below to a different local sample and rerun the parsing + review cells.


In [6]:
# ALT_SAMPLE_DOC = TESTS_DIR / "doc_samples" / "new_test" / "01. 대중문화예술분야 연습생 표준계약서.hwpx"
ALT_SAMPLE_DOC = TESTS_DIR / "doc_samples" / "new_test" / "02. 청소년 대중문화예술인 표준 부속합의서.hwpx"

ALT_SAMPLE_DOC


PosixPath('/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/new_test/02. 청소년 대중문화예술인 표준 부속합의서.hwpx')

## Step 5 - Run the parser LangGraph and color parser labels

This final section runs `doc_processor.run_parser(...)`, reads parser metadata from `working_doc.meta.parser_doc` and `paragraph.meta.parser`, then converts parser spans/categories into exact-text annotations.

The default config below keeps LLM review disabled so the notebook runs without cloud credentials. Turn `label_review_enabled` and `boundary_review_enabled` on when you want the reviewed parser path.


In [7]:
parser_state = run_parser(
    ALT_SAMPLE_DOC,
    config=ParserConfig(
        relevance_mode=RelevanceMode.DISABLED,
        boundary_review_enabled=True,
        label_review_enabled=True,
    ),
)

parser_doc = parser_state.working_doc
parser_result = parser_state.parser_result
assert parser_doc is not None
assert parser_result is not None

parser_annotations, parser_skipped = parser_annotations_from_doc(parser_doc)
parser_category_counts = Counter(annotation.label for annotation in parser_annotations)

parser_review = render_review_html(
    RenderReviewHtmlRequest(
        document=DocumentInput(doc_ir=parser_doc),
        annotations=parser_annotations,
        title="Parser Category Review",
    )
)

parser_review_path = OUTPUT_DIR / f"{ALT_SAMPLE_DOC.stem}_parser_categories.html"
parser_review_path.write_text(parser_review.html or "", encoding="utf-8")

parser_legend = {
    parser_category_label(category): color
    for category, color in PARSER_CATEGORY_COLORS.items()
}

{
    "sample_doc": str(ALT_SAMPLE_DOC.relative_to(TESTS_DIR)),
    "accepted": parser_result.accepted,
    "reason": parser_result.reason,
    "clause_count": parser_result.clause_count,
    "subclause_count": parser_result.subclause_count,
    "annotation_count": len(parser_annotations),
    "category_counts": dict(sorted(parser_category_counts.items())),
    "category_colors": parser_legend,
    "skipped_mixed_content_paragraph_count": len(parser_skipped),
    "skipped_mixed_content_examples": parser_skipped[:10],
    "parser_notes": list(parser_result.notes),
    "parser_review_path": str(parser_review_path.relative_to(TESTS_DIR)),
    "resolved_annotation_preview": [item.model_dump() for item in parser_review.resolved_annotations[:12]],
}


2026-04-15 15:13:35,233 | INFO | structure analysis run start source=/home/maxjo/Work/LAS-system/apps/backend/doc_processor/tests/doc_samples/new_test/02. 청소년 대중문화예술인 표준 부속합의서.hwpx
2026-04-15 15:13:35,312 | INFO | [structure_analysis.load_document] start
2026-04-15 15:13:35,322 | INFO | [structure_analysis.load_document] done goto=screen_relevance
2026-04-15 15:13:35,331 | INFO | [structure_analysis.screen_relevance] start
2026-04-15 15:13:35,332 | INFO | [structure_analysis.screen_relevance] done goto=regex_analysis
2026-04-15 15:13:35,334 | INFO | [structure_analysis.regex_analysis] start
2026-04-15 15:13:35,336 | INFO | [structure_analysis.regex_analysis] done goto=llm_analysis
2026-04-15 15:13:35,339 | INFO | [structure_analysis.llm_analysis] start
2026-04-15 15:13:35,339 | INFO | [structure_analysis.llm_analysis] dispatching boundary batch review (18 suspects)
2026-04-15 15:13:35,340 | INFO | [structure_analysis.llm_analysis] done goto=boundary_llm_batch
2026-04-15 15:13:35,342 | 

{'sample_doc': 'doc_samples/new_test/02. 청소년 대중문화예술인 표준 부속합의서.hwpx',
 'accepted': True,
 'reason': 'Parser clause parsing completed.',
 'clause_count': 16,
 'subclause_count': 17,
 'annotation_count': 54,
 'category_counts': {'Appendix': 1,
  'Clause Body': 1,
  'Clause Heading': 16,
  'Input Block': 15,
  'Other': 1,
  'Subclause Body': 2,
  'Subclause Heading': 17,
  'Title': 1},
 'category_colors': {'Title': '#D9EAD3',
  'Preamble': '#FFF2CC',
  'Clause Heading': '#F4CCCC',
  'Clause Body': '#FCE5CD',
  'Subclause Heading': '#D9D2E9',
  'Subclause Body': '#CFE2F3',
  'Input Block': '#D0E0E3',
  'Appendix': '#EAD1DC',
  'Header': '#EFEFEF',
  'Footer': '#EFEFEF',
  'Other': '#E6E6E6',
  'Boundary Suspect': '#F9CB9C'},
 'skipped_mixed_content_paragraph_count': 3,
 'skipped_mixed_content_examples': ['s1.p4', 's1.p6', 's1.p44'],
 'parser_notes': ["Detected clause rule 'article' and subclause rule 'circled'."],
 'parser_review_path': 'results/docir_api_demo/02. 청소년 대중문화예술인 표준 부속합의서_parse

Failed to export span batch due to timeout, max retries or shutdown.
